# 04 — Recourse Dataset + Level 1: Prediction Fairness

**Author:** Matteo

Integrates with the team's pipeline:
  - Reads `.npy` splits from `data/processed/` (output of 1_Preprocessing.ipynb)
  - Reads model artifacts from `data/models/RF/` and `data/models/XGBoost/`
    (output of 2_Random_Forest.ipynb and 2_XGboost.ipynb)
  - Reads counterfactual setup from utils.py (reuses Kamila's DiCE functions)
  - Saves results to `data/fairness/`

## 0. Setup — install dependencies and load utils

In [ ]:
# Install missing packages (Colab)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dice-ml", "xgboost", "-q"])

# Download utils.py from the repo if not already present
import os
if not os.path.exists("utils.py"):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/francescagrasso02/TechniquesOfAI/main/utils.py",
        "utils.py"
    )
    print("utils.py downloaded.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dice_ml

import utils
from utils import (
    ScaledModelWrapper, load_artifacts, load_raw_with_split,
    build_recourse_dataset, get_project_paths,
    PROTECTED_ATTRIBUTES, SEED_DICE,
)

# ── Paths: auto-detects Colab/Drive or local repo ────────────────────────────
PATHS = get_project_paths()
os.makedirs(PATHS["FAIRNESS"], exist_ok=True)

print("Paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

THRESHOLD = 0.20
N_CF      = 3

## 1. Load data (readable demographics) and model artifacts

In [ ]:
df_raw, df_encoded, feature_cols = load_raw_with_split()
print(f"Dataset: {df_raw.shape[0]} employees, {len(feature_cols)} features")
print(f"True attrition rate: {df_raw['Attrition'].mean():.1%}")

In [ ]:
rf  = load_artifacts(PATHS["RF"])
xgb = load_artifacts(PATHS["XGBoost"])
feature_names = rf["feature_names"]

rf_wrapper  = ScaledModelWrapper(rf["model"],  rf["scaler"],  feature_names)
xgb_wrapper = ScaledModelWrapper(xgb["model"], xgb["scaler"], feature_names)
print("Models loaded successfully.")

## 2. Build DiCE explainers (reusing the team's exact configuration)

In [ ]:
dice_df = df_encoded[feature_names].astype("float64").copy()
dice_df["Attrition"] = df_encoded["Attrition"].values

dice_data = dice_ml.Data(
    dataframe=dice_df,
    continuous_features=list(feature_names),
    outcome_name="Attrition",
)
dice_rf  = dice_ml.Model(model=rf_wrapper,  backend="sklearn", model_type="classifier")
dice_xgb = dice_ml.Model(model=xgb_wrapper, backend="sklearn", model_type="classifier")
exp_rf  = dice_ml.Dice(dice_data, dice_rf,  method="random")
exp_xgb = dice_ml.Dice(dice_data, dice_xgb, method="random")
print("DiCE explainers ready.")

## 3. Generate recourse dataset on ALL at-risk employees

⚠️ This is the slow step (~5-10 min). Results are checkpointed every 25
employees so a crash never loses all progress.

In [ ]:
recourse_rf = build_recourse_dataset(
    exp_rf, rf_wrapper, df_encoded, df_raw, feature_names,
    le_dict=rf["label_encoders"], n_cf=N_CF, seed=SEED_DICE,
    threshold=THRESHOLD,
    checkpoint_path=os.path.join(PATHS["FAIRNESS"], "recourse_RF.csv"),
)

In [ ]:
recourse_xgb = build_recourse_dataset(
    exp_xgb, xgb_wrapper, df_encoded, df_raw, feature_names,
    le_dict=xgb["label_encoders"], n_cf=N_CF, seed=SEED_DICE,
    threshold=THRESHOLD,
    checkpoint_path=os.path.join(PATHS["FAIRNESS"], "recourse_XGB.csv"),
)

print(f"\nDone. RF: {len(recourse_rf)} employees | XGB: {len(recourse_xgb)} employees")
recourse_rf.head()

## 4. Level 1 — Prediction Fairness

Three group-fairness notions (from the lecture slides):
- **Statistical parity:** same predicted-positive rate per group
- **Equal opportunity:** same recall (TPR) per group
- **Predictive equality:** same FPR per group

In [ ]:
def prediction_fairness(wrapper, df_encoded, df_raw, feature_names,
                        protected_attr, threshold=THRESHOLD):
    X      = df_encoded[feature_names].astype("float64")
    proba  = wrapper.predict_proba(X)[:, 1]
    y_pred = (proba >= threshold).astype(int)
    y_true = df_raw["Attrition"].values
    rows   = []
    for g in sorted(df_raw[protected_attr].dropna().unique()):
        mask = (df_raw[protected_attr] == g).values
        yt, yp = y_true[mask], y_pred[mask]
        pos, neg = yt == 1, yt == 0
        rows.append({
            protected_attr:        g,
            "n":                   int(mask.sum()),
            "pred_positive_rate":  round(yp.mean(), 3),
            "TPR_recall":          round(yp[pos].mean(), 3) if pos.any() else float("nan"),
            "FPR":                 round(yp[neg].mean(), 3) if neg.any() else float("nan"),
        })
    return pd.DataFrame(rows)

In [ ]:
for model_name, wrapper in [("Random Forest", rf_wrapper), ("XGBoost", xgb_wrapper)]:
    print(f"\n{'='*60}\n{model_name} — Prediction Fairness\n{'='*60}")
    for attr in PROTECTED_ATTRIBUTES:
        tbl = prediction_fairness(wrapper, df_encoded, df_raw, feature_names, attr)
        gaps = {m: round(tbl[m].max() - tbl[m].min(), 3)
                for m in ["pred_positive_rate", "TPR_recall", "FPR"]}
        print(f"\n-- {attr} --")
        print(tbl.to_string(index=False))
        print(f"   gaps (max-min): {gaps}")

## 5. Visualise predicted attrition rate by group

In [ ]:
fig, axes = plt.subplots(1, len(PROTECTED_ATTRIBUTES),
                         figsize=(5 * len(PROTECTED_ATTRIBUTES), 4))
for ax, attr in zip(axes, PROTECTED_ATTRIBUTES):
    rf_tbl  = prediction_fairness(rf_wrapper,  df_encoded, df_raw, feature_names, attr)
    xgb_tbl = prediction_fairness(xgb_wrapper, df_encoded, df_raw, feature_names, attr)
    x = np.arange(len(rf_tbl))
    ax.bar(x - 0.2, rf_tbl["pred_positive_rate"],  0.4, label="RF",  color="#2196F3")
    ax.bar(x + 0.2, xgb_tbl["pred_positive_rate"], 0.4, label="XGB", color="#FF9800")
    ax.set_xticks(x)
    ax.set_xticklabels(rf_tbl[attr], rotation=20, ha="right")
    ax.set_title(f"Predicted attrition rate — {attr}")
    ax.set_ylabel("rate")
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PATHS["FAIRNESS"], "level1_prediction_fairness.png"), dpi=120)
plt.show()
print("Level 1 done. Proceed to 05_counterfactual_fairness.ipynb")